In [58]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

RIOT_PERSONAL_API_KEY = os.getenv("LOL_PERSONAL_API_KEY")
POSTGRES_LOCAL_URL = os.getenv("POSTGRES_LOCAL_URL")

In [59]:
import re

_camel_case_regex = re.compile(
    r"(?<=[a-z])(?=[A-Z])"  # camelCase: survive|Min
    r"|(?<=[A-Z])(?=[A-Z][a-z])"  # acronym: HTML|Parser
    r"|(?<=[a-zA-Z])(?=\d)"  # letter→digit: survive|15
    r"|(?<=\d)(?=[a-zA-Z])"  # digit→letter: 15|min
)
def camel_to_snake(input_str: str) -> str:
    return _camel_case_regex.sub("_", input_str).lower()

def snake_all_keys(json_obj: dict):
    if isinstance(json_obj, dict):
        return {camel_to_snake(key): snake_all_keys(value) for key, value in json_obj.items()}

    if isinstance(json_obj, list):
        return [snake_all_keys(value) for value in json_obj]

    return json_obj


In [60]:
import requests

BASE_URL = "https://eun1.api.riotgames.com"
REGION = "EUN1"

headers = {
    'X-Riot-Token': RIOT_PERSONAL_API_KEY
}

# Fetch league entries (100 players)
page = 1
league_entries = []
url = f"{BASE_URL}/lol/league/v4/entries/RANKED_SOLO_5x5/BRONZE/I"

response = requests.get(url, headers=headers, params={"page": page})
league_entries = snake_all_keys(response.json())
league_entries


[{'queue_type': 'RANKED_SOLO_5x5',
  'tier': 'BRONZE',
  'rank': 'I',
  'puuid': 'mqmxesVc1z8HPinnVx147ZPFsuscF_9o7Mvt5TU3wqACq8QgK3wY8SehOqeSzj8vXVC7iQR7IPRTEg',
  'league_points': 31,
  'wins': 39,
  'losses': 55,
  'veteran': False,
  'inactive': False,
  'fresh_blood': False,
  'hot_streak': False},
 {'queue_type': 'RANKED_SOLO_5x5',
  'tier': 'BRONZE',
  'rank': 'I',
  'puuid': 'BT_wdnurdMIuNQPWyYqmqg2jNkSmA72jPZuCf9FzhJrNPbx5ovjNCocEiVQkeGRNch9khuozk9F88Q',
  'league_points': 92,
  'wins': 34,
  'losses': 46,
  'veteran': False,
  'inactive': False,
  'fresh_blood': False,
  'hot_streak': False},
 {'queue_type': 'RANKED_SOLO_5x5',
  'tier': 'BRONZE',
  'rank': 'I',
  'puuid': 'eY7rtHuzyNKBisYd7qbwu96aO8f810NhYpexEL6Tg4V1oxeKK0fWnCS_hsI1oAmRKMDbP3T1BwvuDg',
  'league_points': 73,
  'wins': 24,
  'losses': 27,
  'veteran': False,
  'inactive': False,
  'fresh_blood': False,
  'hot_streak': False},
 {'queue_type': 'RANKED_SOLO_5x5',
  'tier': 'BRONZE',
  'rank': 'I',
  'puuid': 'szB

In [61]:
import polars as pl

pl.DataFrame(league_entries)

queue_type,tier,rank,puuid,league_points,wins,losses,veteran,inactive,fresh_blood,hot_streak
str,str,str,str,i64,i64,i64,bool,bool,bool,bool
"""RANKED_SOLO_5x5""","""BRONZE""","""I""","""mqmxesVc1z8HPinnVx147ZPFsuscF_…",31,39,55,false,false,false,false
"""RANKED_SOLO_5x5""","""BRONZE""","""I""","""BT_wdnurdMIuNQPWyYqmqg2jNkSmA7…",92,34,46,false,false,false,false
"""RANKED_SOLO_5x5""","""BRONZE""","""I""","""eY7rtHuzyNKBisYd7qbwu96aO8f810…",73,24,27,false,false,false,false
"""RANKED_SOLO_5x5""","""BRONZE""","""I""","""szB9EfJCkRORJ2Ucfr_CjDzPyPEVns…",47,39,50,false,false,false,false
"""RANKED_SOLO_5x5""","""BRONZE""","""I""","""4E7V-kcpBNrDkruzpfp9vh0hazjhep…",29,5,17,false,false,false,false
…,…,…,…,…,…,…,…,…,…,…
"""RANKED_SOLO_5x5""","""BRONZE""","""I""","""t8e9pwjeWcB0cJbj9WyI2thAPbVV6B…",45,29,23,false,false,false,false
"""RANKED_SOLO_5x5""","""BRONZE""","""I""","""h_LX8ms9Vcsr_tqaNfoSMxXp_2Qkun…",60,4,4,false,false,false,false
"""RANKED_SOLO_5x5""","""BRONZE""","""I""","""TwcBO40H0pAc69baMIXZLmrAtvR3s9…",1,37,44,false,false,false,false


In [62]:
import polars as pl
from datetime import date

def parse_league_entry(entry: dict) -> dict:
    return {
        "puuid": entry.get("puuid"), # Note: Not returned by the league entries API directly, requires enrichment
        "queue_type": entry.get("queue_type"),
        "snapshot_date": date.today(),
        "tier": entry.get("tier"),
        "rank": entry.get("rank"),
        "league_points": entry.get("league_points"),
        "wins": entry.get("wins"),
        "losses": entry.get("losses"),
        "hot_streak": entry.get("hot_streak"),
        "veteran": entry.get("veteran"),
        "fresh_blood": entry.get("fresh_blood"),
        "inactive": entry.get("inactive")
    }

def league_entries_to_df(entries: list[dict]) -> pl.DataFrame:
    rows = [parse_league_entry(e) for e in entries]
    return (
        pl.DataFrame(rows)
        .with_columns(
            pl.col("puuid").cast(pl.Utf8),
            pl.col("queue_type").cast(pl.Utf8),
            pl.col("snapshot_date").cast(pl.Date),
            pl.col("tier").cast(pl.Utf8),
            pl.col("rank").cast(pl.Utf8),
            pl.col("league_points").cast(pl.Int32),
            pl.col("wins").cast(pl.Int32),
            pl.col("losses").cast(pl.Int32),
            pl.col("hot_streak").cast(pl.Boolean),
            pl.col("veteran").cast(pl.Boolean),
            pl.col("fresh_blood").cast(pl.Boolean),
            pl.col("inactive").cast(pl.Boolean)
        )
    )

df = league_entries_to_df(league_entries)
df


puuid,queue_type,snapshot_date,tier,rank,league_points,wins,losses,hot_streak,veteran,fresh_blood,inactive
str,str,date,str,str,i32,i32,i32,bool,bool,bool,bool
"""mqmxesVc1z8HPinnVx147ZPFsuscF_…","""RANKED_SOLO_5x5""",2026-05-31,"""BRONZE""","""I""",31,39,55,false,false,false,false
"""BT_wdnurdMIuNQPWyYqmqg2jNkSmA7…","""RANKED_SOLO_5x5""",2026-05-31,"""BRONZE""","""I""",92,34,46,false,false,false,false
"""eY7rtHuzyNKBisYd7qbwu96aO8f810…","""RANKED_SOLO_5x5""",2026-05-31,"""BRONZE""","""I""",73,24,27,false,false,false,false
"""szB9EfJCkRORJ2Ucfr_CjDzPyPEVns…","""RANKED_SOLO_5x5""",2026-05-31,"""BRONZE""","""I""",47,39,50,false,false,false,false
"""4E7V-kcpBNrDkruzpfp9vh0hazjhep…","""RANKED_SOLO_5x5""",2026-05-31,"""BRONZE""","""I""",29,5,17,false,false,false,false
…,…,…,…,…,…,…,…,…,…,…,…
"""t8e9pwjeWcB0cJbj9WyI2thAPbVV6B…","""RANKED_SOLO_5x5""",2026-05-31,"""BRONZE""","""I""",45,29,23,false,false,false,false
"""h_LX8ms9Vcsr_tqaNfoSMxXp_2Qkun…","""RANKED_SOLO_5x5""",2026-05-31,"""BRONZE""","""I""",60,4,4,false,false,false,false
"""TwcBO40H0pAc69baMIXZLmrAtvR3s9…","""RANKED_SOLO_5x5""",2026-05-31,"""BRONZE""","""I""",1,37,44,false,false,false,false


In [64]:
df.write_database(
    table_name="ranks",
    connection=POSTGRES_LOCAL_URL,
    if_table_exists="append",
    engine="adbc"
)

ProgrammingError: INVALID_ARGUMENT: [libpq] Failed to execute COPY statement: PGRES_FATAL_ERROR ERROR:  insert or update on table "ranks" violates foreign key constraint "ranks_puuid_snapshot_date_fkey"
DETAIL:  Key (puuid, snapshot_date)=(mqmxesVc1z8HPinnVx147ZPFsuscF_9o7Mvt5TU3wqACq8QgK3wY8SehOqeSzj8vXVC7iQR7IPRTEg, 2026-05-31) is not present in table "player_snapshots".
. SQLSTATE: 23503